In [ ]:
# Jupyter: run %pip install rdflib python-docx pandas first, if needed.
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

import pandas as pd
from docx import Document
from IPython.display import display
from rdflib import Graph, URIRef, Literal
from rdflib.namespace import RDF, RDFS

WORD_FILE = Path('quantitative_research_People.docx')
DATASET_FILE = Path('wendangWriteDataset.ttl')
REPORT_FILE = Path('viaf_ulan_matching_report.csv')
OUTPUT_FILE = Path('dataset_with_VIAF_ULAN.ttl')

# First run with False to inspect the matching report
# Set True and rerun only after reviewing the matches
SAVE_ENRICHED_DATASET = True

MANUAL_MATCHES = {}

WDT = 'http://www.wikidata.org/prop/direct/'
VIAF_PROPERTY = URIRef(WDT + 'P214')
ULAN_PROPERTY = URIRef(WDT + 'P245')
PERSON_TYPES = {
    URIRef('http://www.wikidata.org/entity/Q215627'),
    URIRef('https://www.wikidata.org/entity/Q215627'),
    URIRef('http://www.wikidata.org/entity/Q5'),
    URIRef('https://www.wikidata.org/entity/Q5'),
}


def normalise_name(value):
    text = unicodedata.normalize('NFKD', str(value)).casefold()
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return re.sub(r'[^\w\u3400-\u9fff]', '', text).replace('_', '')


def name_variants(name):
    """Accept full bilingual name, Latin part, or Chinese part (2+ characters)."""
    variants = {normalise_name(name)}
    latin_part = re.split(r'[\u3400-\u9fff]', name, maxsplit=1)[0].strip()
    han_part = ''.join(re.findall(r'[\u3400-\u9fff]+', name))
    if latin_part:
        variants.add(normalise_name(latin_part))
    if len(han_part) >= 2:
        variants.add(normalise_name(han_part))
    return variants - {''}


def identifier(cell):
    """Return a digit-string ID, ignoring annotations and × in Word cells."""
    value = re.sub(r'\s+', '', str(cell).strip())
    if not value or value[0] in {'×', 'x', 'X', '-', '/'}:
        return ''
    m = re.match(r'^(\d+)', value)
    return m.group(1) if m else ''


def qid_from_text(text):
    match = re.search(r'\bQ\d+\b', str(text), flags=re.I)
    return match.group(0).upper() if match else ''


# ---- Read the existing Word table --------------------------------------------
doc = Document(WORD_FILE)
rows = []
for row in doc.tables[0].rows[1:]:  # columns: PERSON, ROLE, WD, ULAN, VIAF
    cells = [c.text.strip() for c in row.cells]
    if len(cells) < 5 or not cells[0]:
        continue
    rows.append({
        'person': cells[0],
        'qid': qid_from_text(cells[2]),
        'ulan': identifier(cells[3]),
        'viaf': identifier(cells[4]),
    })

entries = [r for r in rows if r['viaf'] or r['ulan']]
print(f'Word document: {len(rows)} people; {len(entries)} with VIAF and/or ULAN.')

graph = Graph()
graph.parse(DATASET_FILE, format='turtle')
original_size = len(graph)

people = set()
for person_type in PERSON_TYPES:
    people.update(graph.subjects(RDF.type, person_type))
people = {person for person in people if isinstance(person, URIRef)}
if not people:
    raise ValueError('No people found with wd:Q215627 or wd:Q5. Check your RDF types.')

by_qid = defaultdict(set)
by_name = defaultdict(set)

for person in people:
    for label in graph.objects(person, URIRef(WDT + 'P2561')):
        for variant in name_variants(str(label)):
            by_name[variant].add(person)

    for predicate, value in graph.predicate_objects(person):
        # Accept both http and https variants of write:hasWikidataResource.
        local_name = str(predicate).rsplit('#', 1)[-1].rsplit('/', 1)[-1]
        if local_name == 'hasWikidataResource':
            qid = qid_from_text(value)
            if qid:
                by_qid[qid].add(person)

print(f'RDF dataset: {original_size:,} triples; {len(people)} person resources.')

report = []
proposed = set()

for entry in entries:
    name, qid = entry['person'], entry['qid']

    if name in MANUAL_MATCHES:
        candidate_set = {URIRef(MANUAL_MATCHES[name])}
        method = 'manual'
    elif qid and by_qid.get(qid):
        candidate_set = by_qid[qid]
        method = 'Wikidata QID'
    else:
        candidate_set = set()
        for variant in name_variants(name):
            candidate_set.update(by_name.get(variant, set()))
        method = 'exact name'

    candidate_set = candidate_set & people
    resource = next(iter(candidate_set)) if len(candidate_set) == 1 else None
    status = 'matched' if resource else ('ambiguous' if candidate_set else 'not found')
    viaf_result = ulan_result = ''

    if resource:
        for source_column, predicate in [
            ('viaf', VIAF_PROPERTY),
            ('ulan', ULAN_PROPERTY),
        ]:
            new_id = entry[source_column]
            if not new_id:
                continue

            existing = {str(o) for o in graph.objects(resource, predicate)}
            if existing and new_id not in existing:
                outcome = f'CONFLICT (existing: {", ".join(sorted(existing))})'
            elif new_id in existing:
                outcome = 'already present'
            else:
                outcome = 'add'
                proposed.add((resource, predicate, Literal(new_id)))

            if source_column == 'viaf':
                viaf_result = outcome
            else:
                ulan_result = outcome

    report.append({
        **entry,
        'status': status,
        'match_method': method,
        'write_resource': str(resource) if resource else '',
        'candidates': ' | '.join(sorted(map(str, candidate_set))),
        'viaf_action': viaf_result,
        'ulan_action': ulan_result,
    })

report_df = pd.DataFrame(report)
report_df.to_csv(REPORT_FILE, index=False, encoding='utf-8-sig')

print('\nMatching results:')
print(report_df['status'].value_counts().to_string())
print(f'New triples proposed: {len(proposed)}')
print(f'Review the report: {REPORT_FILE}')

problem_rows = report_df[
    (report_df['status'] != 'matched') |
    report_df['viaf_action'].str.startswith('CONFLICT') |
    report_df['ulan_action'].str.startswith('CONFLICT')
]
if not problem_rows.empty:
    display(problem_rows[['person', 'qid', 'status', 'candidates',
                          'viaf_action', 'ulan_action']])


if SAVE_ENRICHED_DATASET:

    new_triples = sorted(
        (triple for triple in proposed if triple not in graph),
        key=lambda t: tuple(str(x) for x in t)
    )

    original_content = DATASET_FILE.read_bytes()

    additions = "\n\n# VIAF and ULAN identifiers\n"

    for subject, predicate, value in new_triples:
        additions += (
            f"{subject.n3()} "
            f"{predicate.n3()} "
            f"{value.n3()} .\n"
        )

    with open(OUTPUT_FILE, "wb") as f:
        f.write(original_content)
        f.write(additions.encode("utf-8"))

    check = Graph()
    check.parse(OUTPUT_FILE, format="turtle")

    print(f"Original triples: {original_size}")
    print(f"Triples added: {len(new_triples)}")
    print(f"Final triples: {len(check)}")
    print(f"Saved to: {OUTPUT_FILE}")

else:
    print("Dataset unchanged. Set SAVE_ENRICHED_DATASET = True to save.")
